# Deploy NVIDIA-Nemotron-3-Super-120B-A12B-FP8 on an Amazon SageMaker AI Inference Component (ml.p6-b300.48xlarge)

This example deploys [nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8](https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8) — the FP8 release of NVIDIA's 120B-parameter (12B active) hybrid Mamba-2 / Transformer / Mixture-of-Experts model — on a SageMaker AI **Inference Component** backed by an `ml.p6-b300.48xlarge` (8x NVIDIA Blackwell Ultra B300, 8x288 GB), using **vLLM** and `boto3`.

What this notebook adds over a plain endpoint deployment:

- **Inference Components (ICs).** The endpoint is created *without* a model; the model attaches as a component with an explicit accelerator claim. At tensor parallelism 1 the FP8 checkpoint (~128 GB) fits on one B300, so one instance can host up to **8 independent copies** of the model (`CopyCount`), each a full vLLM engine — the IC equivalent of data parallelism, with SageMaker routing requests across copies.
- **Speculative decoding.** The checkpoint ships one Multi-Token-Prediction (MTP) layer; vLLM uses it as a drafter (`{"method": "mtp", "num_speculative_tokens": 3}`).
- **FP8 end to end.** FP8 weights and activations (NVIDIA Model Optimizer) plus an FP8 KV cache.
- **DLC or custom image.** The SageMaker vLLM Deep Learning Container is the default; a switch deploys any custom vLLM image instead (what it must provide is spelled out below).
- **Model from the Hub or from S3.** Point `model_s3_uri` at an uncompressed S3 prefix to deploy a fine-tuned checkpoint of the same architecture; the container loads it from `/opt/ml/model`.

> **Container version matters.** vLLM **0.32.0** is the minimum we verified for this architecture on Blackwell: the `NemotronHForCausalLM` MoE path, the MTP drafter and the FlashInfer FP8 MoE backend all work out of the box on the `0.32.0-gpu-py312-cu130-ubuntu24.04-sagemaker` DLC.

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import sys
import time
import re
import json
import boto3
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")  # client to interact with SageMaker
sm_runtime = boto3.client("sagemaker-runtime")  # client to interact with SageMaker Endpoints

In [ ]:
#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    arn = boto3.client("sts").get_caller_identity()["Arn"]
    return re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", arn)


def _wait_for_resource(describe_fn, name_key, status_key, label, name, sleep_time=60):
    # Poll a SageMaker resource until it leaves the 'Creating' or 'Updating' state.
    progress = ""
    while True:
        status = describe_fn(**{name_key: name})[status_key]
        if status not in ("Creating", "Updating"):
            break
        progress += "."
        clear_output(wait=True)
        print(f"Waiting for '{name}': {progress}")
        time.sleep(sleep_time)
    print(f"{label}: '{name}', Status: '{status}'")


def wait_for_endpoint(endpoint_name: str, sleep_time: int = 60):
    _wait_for_resource(
        sm.describe_endpoint, "EndpointName", "EndpointStatus",
        "Endpoint", endpoint_name, sleep_time,
    )


def wait_for_ic(ic_name: str, sleep_time: int = 60):
    _wait_for_resource(
        sm.describe_inference_component, "InferenceComponentName", "InferenceComponentStatus",
        "IC", ic_name, sleep_time,
    )

In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside of SageMaker Studio
#
role = None

if role is None:
    role = get_sagemaker_role()
print(role)

## Configuration

The FP8 checkpoint is ~128 GB, so the tensor-parallel degree is set by GPU memory, and the number of copies by how many GPUs are left:

| Instance | GPU memory | `tp` | copies per instance |
| :--- | :--- | :--- | :--- |
| `ml.p6-b300.48xlarge` | 8 x 288 GB | 1 | up to 8 |
| `ml.p6-b200.48xlarge` | 8 x 192 GB | 1 | up to 8 |
| `ml.p5en.48xlarge` | 8 x 141 GB | 2 | up to 4 |
| `ml.p5.48xlarge` | 8 x 80 GB | 4 | up to 2 |

`tp=1` gives the best throughput per GPU; a higher `tp` lowers latency per request. The 256K native context is capped to 16,384 tokens for serving — raise it if your prompts need more and you have KV headroom.

Two optional inputs: `model_s3_uri` (an **uncompressed** S3 prefix with the checkpoint files at its root, e.g. a fine-tuned derivative) replaces the Hub download, and `ml_reservation_arn` lets the endpoint draw its instance from a SageMaker **training plan** purchased for inference (pass the plan's ARN, `arn:aws:sagemaker:<region>:<account>:training-plan/<name>`).

In [ ]:
instance = {"type": "ml.p6-b300.48xlarge", "num_gpu": 8}
tp = 1                      # GPUs per model copy (see the table above)
copy_count = 1              # copies to start with; scale up to instance["num_gpu"] // tp later

model_id = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8"
model_s3_uri = None         # e.g. "s3://my-bucket/models/my-finetuned-nemotron/" -> served from /opt/ml/model
ml_reservation_arn = None   # e.g. "arn:aws:sagemaker:us-east-1:123456789012:training-plan/b300-endpoint"

model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
ic_name = f"{model_name}-ic"
variant_name = "v1"
# ~128 GB of weights download at startup (Hub or S3) — allow generous timeouts
download_timeout = 3600
startup_timeout = 3600

## Container

**Default: the SageMaker vLLM DLC** ([docs](https://aws.github.io/deep-learning-containers/vllm/)). Its entrypoint turns every `SM_VLLM_<FLAG>` environment variable into a `vllm serve` flag (`SM_VLLM_MAX_NUM_SEQS=128` → `--max-num-seqs 128`, the value `true` → a bare flag, a JSON object is passed through verbatim), serves on port 8080, and answers SageMaker's `/ping` and `/invocations`.

**Alternative: a custom vLLM image**, for example a newer vLLM than the DLC ships or one with extra kernels. Set `use_custom_image = True` and `custom_image_uri`. SageMaker hosting imposes three requirements the stock `vllm/vllm-openai` images do **not** meet on their own:

1. The image must live in an **ECR repository in the same region** as the endpoint (cross-account is fine, cross-region is not).
2. The manifest must be **Docker schema 2** (`application/vnd.docker.distribution.manifest.v2+json`); OCI manifests are rejected with `Unsupported manifest media type`.
3. SageMaker starts the container as `<image> serve`, so an executable named `serve` must be on the `PATH`. The upstream vLLM image has no entrypoint and no `serve`.

The `serve` script next to this notebook ([custom-image/usr/local/bin/serve](custom-image/usr/local/bin/serve)) covers requirement 3 and reproduces the DLC's `SM_VLLM_*` translation, so the same environment works for both containers. Adding it to an upstream image without Docker, using [crane](https://github.com/google/go-containerregistry/tree/main/cmd/crane):

```bash
# copy the upstream image into your own ECR repo (same region as the endpoint)
crane copy vllm/vllm-openai:v0.32.1 $ACCOUNT.dkr.ecr.$REGION.amazonaws.com/vllm-custom:v0.32.1
# add the serve script as a layer
tar --owner=0 --group=0 -cf serve_layer.tar -C custom-image usr      # custom-image/usr/local/bin/serve
crane append -b $ACCOUNT.dkr.ecr.$REGION.amazonaws.com/vllm-custom:v0.32.1 -f serve_layer.tar \
             -t $ACCOUNT.dkr.ecr.$REGION.amazonaws.com/vllm-custom:v0.32.1-sagemaker
# crane writes an OCI manifest; re-publish it as Docker schema 2 (same layers, new manifest)
crane manifest $ACCOUNT.dkr.ecr.$REGION.amazonaws.com/vllm-custom:v0.32.1-sagemaker > m.json
python3 - <<'EOF'
import json; m = json.load(open("m.json"))
m["mediaType"] = "application/vnd.docker.distribution.manifest.v2+json"
m["config"]["mediaType"] = "application/vnd.docker.container.image.v1+json"
for l in m["layers"]: l["mediaType"] = "application/vnd.docker.image.rootfs.diff.tar.gzip"
m.pop("annotations", None); json.dump(m, open("m_docker.json", "w"))
EOF
aws ecr put-image --repository-name vllm-custom --image-tag v0.32.1-sagemaker \
    --image-manifest-media-type application/vnd.docker.distribution.manifest.v2+json \
    --image-manifest file://m_docker.json
```

**Tuned kernels with the DLC.** You do not need a custom image to use tuned fused-MoE configurations: vLLM reads them from the directory named by `VLLM_TUNED_CONFIG_FOLDER`. Put the JSON files (e.g. `E=512,N=2688,device_name=NVIDIA_B300,dtype=fp8_w8a8.json`) in a `vllm_tuned_configs/` folder inside the model's S3 prefix and set the variable to `/opt/ml/model/vllm_tuned_configs` — the cell below does this automatically when the folder exists. Custom *kernel packages* (vLLM plugins) do need a layer on top of the DLC (`FROM <dlc> ; RUN pip install <plugin>`), which keeps the DLC entrypoint; only features that require a newer vLLM than the DLC ships need a different base image.

In [ ]:
use_custom_image = False
custom_image_uri = "<ACCOUNT>.dkr.ecr.<REGION>.amazonaws.com/vllm-custom:v0.32.1-sagemaker"

# The vLLM DLC lives in the per-region Deep Learning Containers registry.
DLC_ACCOUNTS = {"ap-south-2": "772153158452", "ap-southeast-4": "457447274322", "eu-south-2": "503227376785",
                "eu-central-2": "380420809688", "il-central-1": "780543022126", "me-central-1": "914824155844"}
dlc_account = DLC_ACCOUNTS.get(region, "763104351884")
dlc_image = f"{dlc_account}.dkr.ecr.{region}.amazonaws.com/vllm:0.32.0-gpu-py312-cu130-ubuntu24.04-sagemaker-v1.1"

inference_image = custom_image_uri if use_custom_image else dlc_image
print(inference_image)

## Environment

Everything vLLM needs is passed as `SM_VLLM_*` variables (the DLC — or the `serve` wrapper — maps them to flags):

| Variable | Value | Why |
| :--- | :--- | :--- |
| `SM_VLLM_MODEL` | Hub id, or `/opt/ml/model` when deploying from S3 | what vLLM loads |
| `SM_VLLM_TENSOR_PARALLEL_SIZE` | `tp` | GPUs per copy |
| `SM_VLLM_MAX_MODEL_LEN` | `16384` | context cap for serving |
| `SM_VLLM_MAX_NUM_SEQS` | `128` | concurrent sequences per copy |
| `SM_VLLM_KV_CACHE_DTYPE` | `fp8` | matches the checkpoint's FP8 KV-cache scheme |
| `SM_VLLM_SPECULATIVE_CONFIG` | `{"method": "mtp", "num_speculative_tokens": 3}` | speculative decoding with the built-in MTP layer |
| `SM_VLLM_LOAD_FORMAT` | `fastsafetensors` | faster weight loading (GPU-direct where available) |
| `SM_VLLM_GPU_MEMORY_UTILIZATION` | `0.90` | share of GPU memory for weights + KV cache |

vLLM's single MTP layer is reused for all three draft positions, so expect per-position acceptance to fall off (roughly 0.75 / 0.50 / 0.30) and a mean acceptance length around 2.5 tokens per step on general prompts. Lower `num_speculative_tokens` to 2 if your traffic is heavily batched, or drop the variable to serve without speculation.

In [ ]:
common_env = {
    "HF_TOKEN": "<YOUR_TOKEN>",  # not required for this public checkpoint, but raises HF rate limits
}

vllm_env = {
    "SM_VLLM_MODEL": "/opt/ml/model" if model_s3_uri else model_id,
    "SM_VLLM_SERVED_MODEL_NAME": "nemotron-3-super",       # the "model" field clients send
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(tp),
    "SM_VLLM_MAX_MODEL_LEN": "16384",
    "SM_VLLM_MAX_NUM_SEQS": "128",
    "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.90",
    "SM_VLLM_KV_CACHE_DTYPE": "fp8",
    "SM_VLLM_LOAD_FORMAT": "fastsafetensors",
    "SM_VLLM_SPECULATIVE_CONFIG": json.dumps({"method": "mtp", "num_speculative_tokens": 3}),
    # Uncomment to expose the model's reasoning channel as a separate field in responses
    # "SM_VLLM_REASONING_PARSER": "nemotron_v3",
}

# Tuned fused-MoE configs shipped inside the S3 model prefix (see "Tuned kernels with the DLC")
if model_s3_uri:
    s3 = boto3.client("s3")
    bucket, prefix = model_s3_uri[5:].split("/", 1)
    tuned = s3.list_objects_v2(Bucket=bucket, Prefix=prefix.rstrip("/") + "/vllm_tuned_configs/", MaxKeys=1)
    if tuned.get("KeyCount"):
        vllm_env["VLLM_TUNED_CONFIG_FOLDER"] = "/opt/ml/model/vllm_tuned_configs"
        print("tuned MoE configs found in the model prefix -> VLLM_TUNED_CONFIG_FOLDER set")

env = common_env | vllm_env

## Deployment

Three steps, in order:
1. **Endpoint config + endpoint** — the variant carries *no* `ModelName`; it only provisions the instance. `ManagedInstanceScaling` and a routing strategy are required for IC-hosting endpoints; `CapacityReservationConfig` is added only when a training plan ARN is given.
2. **Model** — the container, its environment and (for S3 deployments) the uncompressed model data source.
3. **Inference component** — attaches the model to the endpoint with an explicit claim of `tp` accelerators per copy.

In [ ]:
variant = {
    "VariantName": variant_name,
    "InstanceType": instance["type"],
    "InitialInstanceCount": 1,
    "ManagedInstanceScaling": {"Status": "ENABLED", "MinInstanceCount": 1, "MaxInstanceCount": 1},
    "RoutingConfig": {"RoutingStrategy": "LEAST_OUTSTANDING_REQUESTS"},
}
if ml_reservation_arn:
    variant["CapacityReservationConfig"] = {
        "CapacityReservationPreference": "capacity-reservations-only",
        "MlReservationArn": ml_reservation_arn,
    }

_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ExecutionRoleArn=role,
    ProductionVariants=[variant],
)

_ = sm.create_endpoint(EndpointName=endpoint_name,
                       EndpointConfigName=endpoint_config_name)

wait_for_endpoint(endpoint_name)

In [ ]:
container = {"Image": inference_image, "Environment": env}
if model_s3_uri:
    # uncompressed prefix: SageMaker syncs the files to /opt/ml/model before the container starts
    container["ModelDataSource"] = {"S3DataSource": {
        "S3Uri": model_s3_uri, "S3DataType": "S3Prefix", "CompressionType": "None"}}

_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer=container,
)

In [ ]:
_ = sm.create_inference_component(
    InferenceComponentName=ic_name,
    EndpointName=endpoint_name,
    VariantName=variant_name,
    Specification={
        "ModelName": model_name,
        "StartupParameters": {
            "ModelDataDownloadTimeoutInSeconds": download_timeout,
            "ContainerStartupHealthCheckTimeoutInSeconds": startup_timeout,
        },
        "ComputeResourceRequirements": {
            "NumberOfAcceleratorDevicesRequired": tp,
            "NumberOfCpuCoresRequired": 20 * tp,
            "MinMemoryRequiredInMb": 100_000 * tp,
        },
    },
    RuntimeConfig={"CopyCount": copy_count},
)

wait_for_ic(ic_name)

## Inference examples

Requests target the endpoint **plus** the inference component (`InferenceComponentName`). The container serves the OpenAI chat-completions schema; when more than one copy is running, SageMaker routes each request to the copy with the fewest outstanding requests.

### Text inference

In [ ]:
payload = {
    "model": "nemotron-3-super",
    "messages": [
        {"role": "user", "content": "Explain, in three sentences, why a hybrid Mamba-Transformer model can serve long prompts cheaply."}
    ],
    "max_tokens": 300,
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 InferenceComponentName=ic_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()
print(f"✅ Response time: {end_time-start_time:.2f}s\n")

message = response["choices"][0]["message"]
if message.get("reasoning_content"):
    display(Markdown("### Reasoning:\n---"))
    display(Markdown(message["reasoning_content"]))
display(Markdown("### Content:\n---"))
display(Markdown(message["content"]))
print(f"-----------------------\n{response['usage']}")

### Streaming

With `"stream": true` the container returns server-sent events; `invoke_endpoint_with_response_stream` delivers them as they are produced, which is what makes time-to-first-token visible to the client.

In [ ]:
payload = {
    "model": "nemotron-3-super",
    "messages": [{"role": "user", "content": "Write a haiku about speculative decoding."}],
    "max_tokens": 120,
    "stream": True,
    "stream_options": {"include_usage": True},
}

start_time = time.time()
first_token_at = None
res = sm_runtime.invoke_endpoint_with_response_stream(EndpointName=endpoint_name,
                                                      InferenceComponentName=ic_name,
                                                      Body=json.dumps(payload),
                                                      ContentType="application/json")
buffer = b""
for event in res["Body"]:
    buffer += event.get("PayloadPart", {}).get("Bytes", b"")
    while b"\n\n" in buffer:                       # one SSE event per blank-line-terminated block
        block, buffer = buffer.split(b"\n\n", 1)
        for line in block.decode("utf8").splitlines():
            if not line.startswith("data:") or line.strip() == "data: [DONE]":
                continue
            chunk = json.loads(line[5:])
            if chunk.get("choices"):
                delta = chunk["choices"][0]["delta"].get("content") or ""
                if delta and first_token_at is None:
                    first_token_at = time.time()
                print(delta, end="", flush=True)
            elif chunk.get("usage"):
                usage = chunk["usage"]
print(f"\n\n✅ TTFT {first_token_at - start_time:.2f}s, total {time.time() - start_time:.2f}s, usage: {usage}")

## Scaling copies

The instance has `num_gpu // tp` slots. Raising `CopyCount` starts more vLLM engines on the same instance, each claiming `tp` accelerators; SageMaker spreads traffic across them. This is the way to use all eight B300s with `tp=1`.

In [ ]:
target_copies = instance["num_gpu"] // tp     # 8 copies at tp=1 on an 8-GPU instance
if target_copies != copy_count:
    _ = sm.update_inference_component_runtime_config(
        InferenceComponentName=ic_name,
        DesiredRuntimeConfig={"CopyCount": target_copies},
    )
    wait_for_ic(ic_name)
    copy_count = target_copies
print(sm.describe_inference_component(InferenceComponentName=ic_name)["RuntimeConfig"])

## Cleanup

Delete the inference component first — the endpoint cannot be deleted while a component is attached. The instance bills until the endpoint is gone (a training-plan instance returns to the plan's pool).

In [ ]:
_ = sm.delete_inference_component(InferenceComponentName=ic_name)

# wait for the IC to detach before deleting the endpoint
while True:
    try:
        sm.describe_inference_component(InferenceComponentName=ic_name)
        time.sleep(20)
    except sm.exceptions.ClientError:
        break

_ = sm.delete_endpoint(EndpointName=endpoint_name)
_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
_ = sm.delete_model(ModelName=model_name)